In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Se o notebook estiver dentro de RL_cpp/notebooks, isso sobe para RL_cpp/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RUN_NAME = "run_001"

export_dir = PROJECT_ROOT / "runs" / RUN_NAME / "export"
figures_dir = PROJECT_ROOT / "runs" / RUN_NAME / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

map_path = export_dir / "grid_map.json"
q_path = export_dir / "q_table.csv"

print("Project root:", PROJECT_ROOT)
print("Map path:", map_path)
print("Q-table path:", q_path)

In [ ]:
with open(map_path, "r") as f:
    grid_map = json.load(f)

q_df = pd.read_csv(q_path)

height = int(grid_map["height"])
width = int(grid_map["width"])

start = (
    int(grid_map["start"]["row"]),
    int(grid_map["start"]["col"]),
)

goal = (
    int(grid_map["goal"]["row"]),
    int(grid_map["goal"]["col"]),
)

obstacles = {
    (int(obs["row"]), int(obs["col"]))
    for obs in grid_map["obstacles"]
}

print("Grid size:", height, "x", width)
print("Start:", start)
print("Goal:", goal)
print("Number of obstacles:", len(obstacles))

q_df.head()

In [ ]:
V = np.full((height, width), np.nan)
confidence = np.full((height, width), np.nan)

greedy_action = np.full((height, width), -1, dtype=int)
greedy_dr = np.zeros((height, width))
greedy_dc = np.zeros((height, width))

is_obstacle = np.zeros((height, width), dtype=bool)
is_start = np.zeros((height, width), dtype=bool)
is_goal = np.zeros((height, width), dtype=bool)

for _, row in q_df.iterrows():
    r = int(row["row"])
    c = int(row["col"])

    is_obstacle[r, c] = bool(row["is_obstacle"])
    is_start[r, c] = bool(row["is_start"])
    is_goal[r, c] = bool(row["is_goal"])

    V[r, c] = row["value"]
    confidence[r, c] = row["confidence"]

    greedy_action[r, c] = int(row["greedy_action"])
    greedy_dr[r, c] = row["greedy_delta_row"]
    greedy_dc[r, c] = row["greedy_delta_col"]

# Garantir que obstáculos não apareçam no heatmap
V[is_obstacle] = np.nan
confidence[is_obstacle] = np.nan

In [ ]:
def setup_grid_axes(ax, title):
    ax.set_title(title)
    ax.set_aspect("equal")

    ax.set_xlim(-0.5, width - 0.5)
    ax.set_ylim(height - 0.5, -0.5)

    ax.set_xticks(np.arange(width))
    ax.set_yticks(np.arange(height))

    ax.set_xticks(np.arange(-0.5, width, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, height, 1), minor=True)
    ax.grid(which="minor", linewidth=0.5)

    ax.tick_params(which="minor", bottom=False, left=False)


def draw_obstacles(ax):
    for r, c in obstacles:
        rect = plt.Rectangle(
            (c - 0.5, r - 0.5),
            1,
            1,
            fill=True,
            color="black",
            zorder=3,
        )
        ax.add_patch(rect)


def draw_start_goal(ax):
    ax.scatter(
        start[1],
        start[0],
        marker="o",
        s=140,
        color="white",
        edgecolor="black",
        linewidth=1.5,
        label="Start",
        zorder=5,
    )

    ax.scatter(
        goal[1],
        goal[0],
        marker="*",
        s=220,
        color="white",
        edgecolor="black",
        linewidth=1.5,
        label="Goal",
        zorder=5,
    )


def draw_policy_arrows(ax, scale=0.35):
    X, Y = np.meshgrid(np.arange(width), np.arange(height))

    U = greedy_dc * scale
    W = greedy_dr * scale

    mask = (~is_obstacle) & (~is_goal)

    ax.quiver(
        X[mask],
        Y[mask],
        U[mask],
        W[mask],
        angles="xy",
        scale_units="xy",
        scale=1,
        pivot="middle",
        width=0.006,
        zorder=4,
    )

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

empty = np.zeros((height, width))
ax.imshow(empty, origin="upper", alpha=0.0)

draw_obstacles(ax)
draw_start_goal(ax)
setup_grid_axes(ax, "GridWorld environment")

ax.legend(loc="upper right")
plt.show()

fig.savefig(figures_dir / "gridworld_environment.png", dpi=200, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

V_masked = np.ma.array(V, mask=is_obstacle)

im = ax.imshow(V_masked, origin="upper")

draw_obstacles(ax)
draw_policy_arrows(ax)
draw_start_goal(ax)
setup_grid_axes(ax, r"Value heatmap + greedy policy")

cbar = plt.colorbar(im, ax=ax)
cbar.set_label(r"$V(s) = \max_a Q(s,a)$")

ax.legend(loc="upper right")
plt.show()

fig.savefig(figures_dir / "value_heatmap_greedy_policy.png", dpi=200, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

confidence_masked = np.ma.array(confidence, mask=is_obstacle)

im = ax.imshow(confidence_masked, origin="upper")

draw_obstacles(ax)
draw_start_goal(ax)
setup_grid_axes(ax, "Greedy-action confidence heatmap")

cbar = plt.colorbar(im, ax=ax)
cbar.set_label(r"$\max_a Q(s,a) - \mathrm{second\ best}_a Q(s,a)$")

ax.legend(loc="upper right")
plt.show()

fig.savefig(figures_dir / "confidence_heatmap.png", dpi=200, bbox_inches="tight")

In [ ]:
def compute_greedy_path(max_steps=500):
    r, c = start
    path = [(r, c)]
    visited = {(r, c): 0}
    loop_start_index = None

    for _ in range(max_steps):
        if (r, c) == goal:
            break

        dr = int(greedy_dr[r, c])
        dc = int(greedy_dc[r, c])

        nr = r + dr
        nc = c + dc

        if not (0 <= nr < height and 0 <= nc < width):
            break

        if is_obstacle[nr, nc]:
            break

        r, c = nr, nc
        path.append((r, c))

        if (r, c) in visited:
            loop_start_index = visited[(r, c)]
            break

        visited[(r, c)] = len(path) - 1

    return path, loop_start_index


path, loop_start_index = compute_greedy_path(max_steps=500)

print("Path length:", len(path))
print("Reached goal:", path[-1] == goal)
print("Loop detected:", loop_start_index is not None)

if loop_start_index is not None:
    print("Loop starts at index:", loop_start_index)
    print("Loop state:", path[loop_start_index])

fig, ax = plt.subplots(figsize=(9, 8))

V_masked = np.ma.array(V, mask=is_obstacle)
im = ax.imshow(V_masked, origin="upper")

draw_obstacles(ax)
draw_policy_arrows(ax, scale=0.25)
draw_start_goal(ax)

path_rows = [p[0] for p in path]
path_cols = [p[1] for p in path]

ax.plot(path_cols, path_rows, linewidth=2.5, marker="o", markersize=3, label="Greedy path", zorder=6)

if loop_start_index is not None:
    loop_r, loop_c = path[loop_start_index]
    ax.scatter(loop_c, loop_r, marker="x", s=160, linewidth=3, label="Loop start", zorder=7)

setup_grid_axes(ax, "Greedy path over value heatmap")

cbar = plt.colorbar(im, ax=ax)
cbar.set_label(r"$V(s) = \max_a Q(s,a)$")

ax.legend(loc="upper right")
plt.show()

fig.savefig(figures_dir / "greedy_path_value_heatmap.png", dpi=200, bbox_inches="tight")